# Appdetails to CSV

This notebook gathers selected metadata from the individual json files and create one CSV file for all games.

This format can be used to define samples of all games.


In [4]:
import pandas as pd
import csv

# Define metadata fields as columns
metadata_columns = [
    "name",
    "steam_appid",
    "required_age",
    "is_free",
    "number_dlc",
    "developers",
    "publishers",
    "price_currency",
    "price_initial",
    "price_final",
    "windows",
    "mac",
    "linux",
    "metacritic_score",
    "categories",
    "genres",
    "recommendations_total",
    "achievements_total",
    "release_date"
]

# Create an empty DataFrame
df = pd.DataFrame(columns=metadata_columns)


In [5]:
import json

def metadata_json_to_df(file):
    new_item = dict()
    try:
        with file.open('r', encoding='utf-8') as f:
            data = json.load(f)
            data_keys = list(data.keys())

            if "name" in data_keys:
                new_item["name"] = str(data["name"])

            if "steam_appid" in data_keys:
                new_item["steam_appid"] = int(data["steam_appid"])

            if "required_age" in data_keys:
                new_item["required_age"] = int(data["required_age"])

            if "is_free" in data_keys:
                new_item["is_free"] = data["is_free"]

            if "dlc" in data_keys:
                new_item["number_dlc"] = int(len(data["dlc"]))

            if "developers" in data_keys:
                list_dev= ""
                for d in data["developers"]:
                    list_dev += str(d) + ";"
                new_item["developers"] = "" if list_dev == "" else list_dev[:-1]

            if "publishers" in data_keys:
                list_pub = ""
                for p in data["publishers"]:
                    list_pub += str(p) + ";"
                new_item["publishers"] = "" if list_pub == "" else list_pub[:-1]

            if "price_overview" in data_keys:
                if "currency" in data["price_overview"].keys():
                    new_item["price_currency"] = data["price_overview"]["currency"]
                if "initial" in data["price_overview"].keys():
                    new_item["price_initial"] = data["price_overview"]["initial"] * 0.01
                if "final" in data["price_overview"].keys():
                    new_item["price_final"] = data["price_overview"]["final"] * 0.01

            if "platforms" in data_keys:
                if "windows" in data["platforms"].keys():
                    new_item["windows"] = data["platforms"]["windows"]
                if "mac" in data["platforms"].keys():
                    new_item["mac"] = data["platforms"]["mac"]
                if "linux" in data["platforms"].keys():
                    new_item["linux"] = data["platforms"]["linux"]

            if "metacritic" in data_keys:
                if "score" in data["metacritic"].keys():
                    new_item["metacritic_score"] = data["metacritic"]["score"]

            if "categories" in data_keys:
                list_cat = ""
                for c in data["categories"]:
                    list_cat += str(c["description"]) + ";"
                new_item["categories"] = "" if list_cat == "" else list_cat[:-1]

            if "genres" in data_keys:
                list_gen = ""
                for g in data["genres"]:
                    list_gen += str(g["description"]) + ";"
                new_item["genres"] = "" if list_gen == "" else list_gen[:-1]

            if "recommendations" in data_keys:
                if "total" in data["recommendations"].keys():
                    new_item["recommendations_total"] = data["recommendations"]["total"]

            if "achievements" in data_keys:
                if "total" in data["achievements"].keys():
                    new_item["achievements_total"] = data["achievements"]["total"]

            if "release_date" in data_keys:
                if "date" in data["release_date"].keys():
                    new_item["release_date"] = data["release_date"]["date"]

            return new_item
    except Exception as e:
        print(f"Could not read {file.name}: {e}")
        return -1

In [6]:
from pathlib import Path
from tqdm import tqdm

dir_path = Path('./raw_metadata_dataset')
files = [f for f in dir_path.iterdir() if f.is_file()]

for file in tqdm(files[:10000]):
    try:
        item = metadata_json_to_df(file)
        if item:
            df = pd.concat([df, pd.DataFrame([item])], ignore_index=True)
    except Exception as e:
        print(f"Issue while processing {file.name}: {e}")

# Save the DataFrame to a CSV file
df.to_csv('games_metadata.csv',
          quoting=csv.QUOTE_NONNUMERIC,
          quotechar='"',
          sep=",",
          float_format='%.2f',
          index=False)

print("CSV file saved as 'games_metadata.csv'")

  0%|          | 0/10000 [00:00<?, ?it/s]/var/folders/h2/hb2skljd1x7c0k8xq2mxd1yh0000gn/T/ipykernel_12859/1109726909.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([item])], ignore_index=True)
100%|██████████| 10000/10000 [00:11<00:00, 863.60it/s]


CSV file saved as 'games_metadata.csv'
